In [4]:
import cv2
import numpy as np

<p style="color: skyblue; font-size: 30px;">Classification de la couleur des citrons (vert et jaune)</p>

In [16]:
cap = cv2.VideoCapture(0)  #le Videocapture(0)  c'est juste our ouvrir la camera de l'ordi, le 0 est pour la camera par defaut

while True :
    ret, frame = cap.read() # Cette ligne de code capture une image d ela webcam. le read() c'est pour lire les frames de la video, et le ret c'est pour savoir si la lecture a reussi ou pas (re = True si ça a reussi)
    if not ret : #quitte si la camera ne fournit plus d'images
        break

#Ensuite on applique un flou gaussien pour reduire le bruit de l'image et faciliter la detection des contours
    blur=cv2.GaussianBlur(frame, (9,9), 1.5) #Le GaussianBlur() c'est pour appliquer un flou gaussien a l'image, le (9,9) c'est la taille du noyau de convolution, et le 1.5 c'est l'ecart type du flou

    hsv=cv2.cvtColor(blur, cv2.COLOR_BGR2HSV) #Le cvtColor() c'est pour convertir l'image du format BGR (Blue, Green, Red) au format HSV (Hue, Saturation, Value), ce qui facilite la detection des couleurs

    gray =cv2.cvtColor(blur, cv2.COLOR_BGR2GRAY) #Le cvtColor() c'est pour convertir l'image du format HSV au format grayscale (niveaux de gris), ce qui facilite la detection des contours. On fait ça car HoughCircles ne fonctionne pas bien sur les images en couleur


#On passe à la détectin de cercles avec la fonction HoughCircles() de OpenCV, qui utilise la transformée de Hough pour détecter les cercles dans l'image
    circles = cv2.HoughCircles( # HoughCircles détecte les cercles dans l'image  (la seule disponible dans OpenCV)
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist =100, #cv2.HOUGH_GRADIENT → méthode de détection, dp=1.2 → résolution de l’accumulateur (1.0 = même taille que image, >1 = réduit la résolution)  Plus la valeur est haute, moins c'est précis mais plus c'est rapide, minDist=100 → distance minimale entre les centres de cercles détectés
        param1=120, param2=50, minRadius=30, maxRadius=200) #param1=50 → seuil pour le détecteur de bords Canny  (plus élevé = moins sensible), param2=30 → seuil pour valider un cercle dans l’accumulateur Hough (plus bas = détecte plus de cercles, mais aussi plus de faux positifs), minRadius et maxRadius → rayon minimum et maximum des cercles détectés

#Boucle sur chaquee cercles détecté pour analyser sépaement 
    if circles is not None:
        circles = np.uint16(np.around(circles)) #np.around → arrondit les coordonnées, np.uint16 → convertit en entier pour OpenCV
        
        for (x, y, r) in circles[0]:
            mask = np.zeros(gray.shape, dtype=np.uint8) # image noire de la même taille que gray
            cv2.circle(mask, (x, y), r, 255, -1) #cv2.circle() → dessine un cercle blanc (255) sur le masque à la position (x, y) avec le rayon r, -1 signifie que le cercle est rempli, le but de cette partie c'est d'isolé le cercle pour ne prendre que ça couleur
            mean_hsv = cv2.mean(hsv, mask=mask) # calcule la couleur moyenne seulement là où le mask est blanc
            h = mean_hsv[0] #la teinte dominante, qui servira à classer le citron
        
            if 35 <= h <= 100: #color → couleur pour dessiner le cercle et le texte à l’écran
                label = "Cercle VERT"
                color = (0, 255, 0)  # vert
            elif 20 <= h <= 34:
                label = "Cercle JAUNE"
                color = (0, 255, 255)  # jaune
            else:
                label = "INCONNU"
                color = (255, 0, 255)  # rouge
                
            cv2.circle(frame, (x, y), r, color, 3) # dessine un cercle de centre (x,y), de rayon r, color c'est la couleur du cercle, 3 c'est l'epaisseur des bordures, frame c'est l'image sur laquel on doi dessiner le cercle
            cv2.putText(frame, label, (x - 50, y - r - 10), # label c'est le texte a afficher, (x - 50, y - r - 10) c'est la position du texte (un peu au dessus du cercle),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2) # cv2.FONT_HERSHEY_SIMPLEX c'est la police du texte, 0.8 c'est la taille du texte, color c'est la couleur du texte, 2 c'est l'epaisseur du texte
        
    cv2.imshow("Reconnaissance Citron", frame) #affiche le résultat dans une fenêtre intitulée "Reconnaissance Citron"
    
    if cv2.waitKey(1) & 0xFF == 27:  # cv2.waitKey(1) → attend 1 ms pour un événement clavier,27 → touche ESC, Si ESC pressée → quitte la boucle
        break
    
cap.release() #Libère la webcam
cv2.destroyAllWindows() #Ferme toutes les fenêtres OpenCV ouvertes

C:\Users\farou\AppData\Local\Temp\ipykernel_28168\1313504404.py:42: RuntimeWarning: overflow encountered in scalar subtract
  cv2.putText(frame, label, (x - 50, y - r - 10), # label c'est le texte a afficher, (x - 50, y - r - 10) c'est la position du texte (un peu au dessus du cercle),
